In [118]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta
import os
from dotenv import load_dotenv

load_dotenv()

True

In [119]:
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={os.getenv('DB_SERVER')};"
    "DATABASE=Ventas_Comerssia;"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)


# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [120]:
query = """
SELECT DISTINCT v.Pedido,
    v.Cliente,
    c.Nombres + ' ' + c.Apellidos AS Nombre,
    c.Email AS Email,
    c.Celular AS Celular,
    v.Fecha AS Fecha_Venta,
    v.Canal,
    SUM(v.Venta_Neta) AS Venta
FROM Ventas_Comerssia.dbo.Contacto_Clientes c
INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
WHERE v.fecha >= '2026-07-31'
    AND v.Fecha <= '2026-07-31'
    AND (Canal = 'Shopify' OR Ubicacion = 'Social Selling')
    AND v.Venta_Neta > 0
GROUP BY  v.Pedido,
    v.Cliente,
    c.Nombres + ' ' + c.Apellidos,
    c.Email,
    c.Celular,
    v.Fecha,
    v.Canal
"""

# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query, engine)
df_ventas.head()

,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,101962,C1026575005,None,sandraleonrodriguez10@hotmail.com,3204349314,2026-07-31,Shopify,236586.47
1,101963,C1026259695,MILENA ACOSTA,miluac@gmail.com,3102583362,2026-07-31,Shopify,112810.27
2,101964,C66997567,IVONE CARMONA,ivonecarmona05@gmail.com,3153327776,2026-07-31,Shopify,62392.28
3,101965,C1000185915,GABRIELA MARIÑO,gmarinogutierrez@gmail.com,3188204219,2026-07-31,Shopify,265870.92
4,101966,C1007415016,JENNIFER FUENTES,jeniferfuentes43@gmail.com,3209676182,2026-07-31,Shopify,67434.08


In [121]:
# # CONSULTAS CUMPLEAÑOS

# query_data = """
# SELECT DISTINCT v.Pedido,
# 	v.Cliente, 
# 	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
# 	cc.Email, cc.Celular, 
# 	v.Fecha AS Fecha_Venta,
#     Canal,
#     SUM(Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
# INNER JOIN Ventas_Comerssia.dbo.Contacto_Clientes cc ON cc.Cliente = v.Cliente
# WHERE (Codigo_Descuento = '0001 Cumpleaños Cliente 25%' or Codigo_Descuento = 'JULIO_2026')
# 	AND Fecha BETWEEN '2026-07-11' AND '2026-07-16'
# GROUP BY  v.Pedido,
# 	v.Cliente,
#     cc.Nombres + ' ' +  cc.Apellidos,
#     cc.Email,
#     cc.Celular,
#     v.Fecha,
#     v.Canal
# """


# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)
# df_ventas.head()

In [122]:
# Leer archivo
df_indigitall = pd.read_excel("Atribucion.xlsx")

# Seleccionar columnas relevantes
df_campania = df_indigitall[['phone', 'campaignName', 'sentAt',  'clicks']].copy()

# Renombrar columnas
df_campania.rename(columns={
    'phone': 'Celular',
    'campaignName': 'campania',
    'sentAt': 'fecha_envio',
    'clicks': 'clics'
}, inplace=True)

# Extraer primera fecha de apertura (si viene como lista en string)
df_campania['clics'] = df_campania['clics'].apply(
    lambda x: eval(x)[0] if isinstance(x, str) and x.startswith('[') and len(eval(x)) > 0 else None
)
df_campania['clics'] = pd.to_datetime(df_campania['clics'], errors='coerce').dt.tz_localize(None)
df_campania['fecha_envio'] = pd.to_datetime(df_campania['fecha_envio'], errors='coerce').dt.tz_localize(None).dt.date

df_enviados = df_campania[df_campania['fecha_envio'].notnull()].copy()
df_clics =  df_campania[df_campania['clics'].notnull()].copy()

fecha_maxima = pd.to_datetime('2026-07-31').date()
df_clics_filtrados = df_clics[df_clics['clics'].dt.date <= fecha_maxima]

df_clics_filtrados = df_clics_filtrados.drop_duplicates(subset='Celular', keep='first')

total_clics = df_clics_filtrados['Celular'].nunique()


In [123]:
# df_enviados['Celular'] = df_enviados['Celular'].astype(str)
# df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# # Merge por email
# df_merge = pd.merge(df_ventas, df_enviados, on='Celular', how='inner')

# df_merge

In [124]:
df_enviados['Celular'] = df_enviados['Celular'].astype(str)
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por email
df_merge = pd.merge(df_ventas, df_enviados, on='Celular', how='inner')

# Filtrar ventas dentro de 5 días desde la fecha de envio
df_merge['Fecha_Venta'] = pd.to_datetime(df_merge['Fecha_Venta'])
df_merge['fecha_envio'] = pd.to_datetime(df_merge['fecha_envio'])
df_merge['dias_post_apertura'] = (df_merge['Fecha_Venta'] - df_merge['fecha_envio']).dt.days
df_atribucion = df_merge[
    (df_merge['dias_post_apertura'] >= 0) &
    (df_merge['dias_post_apertura'] <= 5)
]

# Sumar la venta total atribuida
tottal_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Número de clientes que hicieron clic hasta {fecha_maxima}: {total_clics}")
print(f"Clientes Efectivos: {filas}")
print(f"Total de venta atribuida: {tottal_venta:,.2f}")
# Resultado final
df_atribucion.head()

Número de clientes que hicieron clic hasta 2026-07-31: 361
Clientes Efectivos: 43
Total de venta atribuida: 11,506,821.58


,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,campania,fecha_envio,clics,dias_post_apertura
0,101964,C66997567,IVONE CARMONA,ivonecarmona05@gmail.com,3153327776,2026-07-31,Shopify,62392.28,PRUEBA: SALE WEB Y SOCIAL,2026-07-31,NaT,0
1,101969,C1032376445,JOHANNA VELA,johanna.vela@gmail.com,3007388607,2026-07-31,Shopify,198520.87,PRUEBA: SALE WEB Y SOCIAL,2026-07-31,NaT,0
2,101977,C63297194,MARIA CLAUDIA ESTEBAN JIMENEZ,mclaudiaesteban@gmail.com,3112615385,2026-07-31,Shopify,555648.37,PRUEBA: SALE WEB Y SOCIAL,2026-07-31,NaT,0
3,101982,C28550956,MERY CONSTANZA DAZA,merycdaza2020@gmail.com,3234980296,2026-07-31,Shopify,129196.13,PRUEBA: SALE WEB Y SOCIAL,2026-07-31,NaT,0
4,101992,C1018411176,ANDRES GALLO,andresfgallo@hotmail.com,3114773850,2026-07-31,Shopify,218057.85,PRUEBA: SALE WEB Y SOCIAL,2026-07-31,NaT,0


In [125]:
df_atribucion.to_excel('Ventas.xlsx', index=False)